In [ ]:
# Day 22 - Advanced RAG Techniques

!pip install -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate

import torch
from transformers import pipeline

In [ ]:
# 1. Better Document Loading & Chunking
text = """
Ethiopia is one of the oldest countries in the world. 
Addis Ababa is its capital and a major economic hub in Africa. 
The country is known for its rich history, diverse cultures, and unique calendar.
Injera is the national dish. Coffee originated in Ethiopia. 
Lalibela is famous for its rock-hewn churches, a UNESCO World Heritage Site.
"""

# Better chunking strategy
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)

documents = text_splitter.create_documents([text])
print(f"Created {len(documents)} chunks with overlap")

In [ ]:
# 2. Embeddings + Vector Store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
# 3. Load LLM
pipe = pipeline("text-generation", 
                model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                device=0 if torch.cuda.is_available() else -1,
                max_new_tokens=300)

llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
# 4. Advanced RAG Prompt
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a knowledgeable Ethiopian AI assistant. Answer the question using only the provided context.

Context:
{context}

Question: {question}
Answer in a clear and friendly way:"""
)

# Simple RAG function
def advanced_rag_query(question):
    docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    prompt = prompt_template.format(context=context, question=question)
    response = llm.invoke(prompt)
    print("Q:", question)
    print("A:", response.strip())
    return response

# Test
advanced_rag_query("What is famous in Lalibela?")
advanced_rag_query("Tell me about Ethiopian food.")